# Recommender systems: collaborative filtering: lab notebook

**Machine Learning (5CS4-05), Recommenders, neural networks, deep learning**

Run the cells from top to bottom (Runtime > Run all). NumPy and scikit-learn are already installed in Colab. Read each section, run it, and compare your output with the expected output at the end of the notebook.

In [ ]:
import numpy as np

users = ["Ann", "Bob", "Cat", "Dan", "Eve", "Fay"]
items = ["Matrix", "Titanic", "Alien", "Notebook", "Avatar", "Up"]
R = np.array([[5, 1, 5, 1, 4, 0],
              [4, 0, 4, 2, 5, 1],
              [1, 5, 2, 5, 0, 4],
              [0, 4, 1, 5, 2, 5],
              [5, 2, 0, 1, 5, 2],
              [2, 5, 1, 0, 1, 4]], dtype=float)           # 0 = not rated

### 1. Similarity over the items both vectors rated

In [ ]:
# 1. Similarity over the items both vectors rated
def sim(a, b, metric="cosine"):
    both = (a > 0) & (b > 0)
    if both.sum() < (2 if metric == "pearson" else 1):
        return 0.0
    x, y = a[both], b[both]
    if metric == "pearson":
        x, y = x - x.mean(), y - y.mean()
    den = np.sqrt((x * x).sum() * (y * y).sum())
    return float(x @ y / den) if den > 0 else 0.0

### 2. User-based prediction: mean-centred weighted average of neighbours

In [ ]:
# 2. User-based prediction: mean-centred weighted average of neighbours
def predict_user(u, i, k=3, metric="cosine"):
    mean = lambda r: r[r > 0].mean()
    cand = [(sim(R[u], R[v], metric), v) for v in range(len(R)) if v != u and R[v, i] > 0]
    top = sorted([c for c in cand if c[0] > 0], reverse=True)[:k]
    if not top:
        return mean(R[u])
    return float(mean(R[u]) + sum(s * (R[v, i] - mean(R[v])) for s, v in top) / sum(s for s, _ in top))

u, i = 0, 5
print("Ann's similarity to the others:", {users[v]: round(sim(R[u], R[v]), 3) for v in range(1, 6)})
print(f"user-based prediction for Ann on Up: {predict_user(u, i):.2f}")

### 3. Item-based prediction: weighted average of the user's own ratings

In [ ]:
# 3. Item-based prediction: weighted average of the user's own ratings
def predict_item(u, i, k=3, metric="cosine"):
    cand = [(sim(R[:, i], R[:, j], metric), j) for j in range(R.shape[1]) if j != i and R[u, j] > 0]
    top = sorted([c for c in cand if c[0] > 0], reverse=True)[:k]
    return float(sum(s * R[u, j] for s, j in top) / sum(s for s, _ in top)) if top else R[u][R[u] > 0].mean()

print(f"item-based prediction for Ann on Up: {predict_item(u, i):.2f}")

### 4. Recommend the unrated items for Ann

In [ ]:
# 4. Recommend the unrated items for Ann
unrated = [j for j in range(6) if R[u, j] == 0]
print("recommendations for Ann:", [(items[j], round(predict_user(u, j), 2)) for j in unrated])

### 5. Matrix factorisation by gradient descent (the idea behind many real systems)

In [ ]:
# 5. Matrix factorisation by gradient descent (the idea behind many real systems)
rng = np.random.default_rng(0)
P, Q = 0.1 * rng.standard_normal((6, 2)), 0.1 * rng.standard_normal((6, 2))
known = np.argwhere(R > 0)
for epoch in range(2000):
    for a, b in known:
        err = R[a, b] - P[a] @ Q[b]
        P[a], Q[b] = P[a] + 0.02 * (err * Q[b] - 0.02 * P[a]), Q[b] + 0.02 * (err * P[a] - 0.02 * Q[b])
rmse = np.sqrt(np.mean([(R[a, b] - P[a] @ Q[b]) ** 2 for a, b in known]))
print("matrix factorisation training RMSE:", round(float(rmse), 3), " predicted Ann-Up (clipped to 1-5):", round(float(np.clip(P[0] @ Q[5], 1, 5)), 2))

## Expected output

Running all cells above should print:

```
Ann's similarity to the others: {'Bob': 0.97, 'Cat': 0.467, 'Dan': 0.495, 'Eve': 0.987, 'Fay': 0.527}
user-based prediction for Ann on Up: 2.24
item-based prediction for Ann on Up: 1.98
recommendations for Ann: [('Up', 2.24)]
matrix factorisation training RMSE: 0.422  predicted Ann-Up (clipped to 1-5): 1.0
```

## Practice exercises

Try these in the cells below.

1. Change the target user to Dan and the item to Alien. Which neighbour has the most influence?
2. Use item-based CF with Pearson similarity and compare the prediction for (Ann, Up).
3. Mask 20% of the known ratings, predict them and report the RMSE for k = 1, 2, 3.

In [ ]:
# Exercise 1


In [ ]:
# Exercise 2


In [ ]:
# Exercise 3
